# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda II

Vamos continuar trabalhando com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('previsao_de_renda.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             15000 non-null  int64  
 1   data_ref               15000 non-null  object 
 2   id_cliente             15000 non-null  int64  
 3   sexo                   15000 non-null  object 
 4   posse_de_veiculo       15000 non-null  bool   
 5   posse_de_imovel        15000 non-null  bool   
 6   qtd_filhos             15000 non-null  int64  
 7   tipo_renda             15000 non-null  object 
 8   educacao               15000 non-null  object 
 9   estado_civil           15000 non-null  object 
 10  tipo_residencia        15000 non-null  object 
 11  idade                  15000 non-null  int64  
 12  tempo_emprego          12427 non-null  float64
 13  qt_pessoas_residencia  15000 non-null  float64
 14  renda                  15000 non-null  float64
dtypes:

1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
2. Rode uma regularização *ridge* com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o $R^2$ na base de testes. Qual o melhor modelo?
3. Faça o mesmo que no passo 2, com uma regressão *LASSO*. Qual método chega a um melhor resultado?
4. Rode um modelo *stepwise*. Avalie o $R^2$ na vase de testes. Qual o melhor resultado?
5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?
6. Partindo dos modelos que você ajustou, tente melhorar o $R^2$ na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.
7. Ajuste uma árvore de regressão e veja se consegue um $R^2$ melhor com ela.

REsolvendo item 01
Separe a base em treinamento e teste (25% para teste, 75% para treinamento).

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('previsao_de_renda.csv')
print("Base carregada:", df.shape)

df = df.drop(columns=['Unnamed: 0','data_ref','id_cliente'])
X = df.drop(columns='renda')
y = np.log(df['renda'])

cat = X.select_dtypes(include=['object','bool']).columns
num = X.select_dtypes(include=['int64','float64']).columns

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('scaler', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))]), cat)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

Base carregada: (15000, 15)
Treino: (11250, 11) | Teste: (3750, 11)


Resolvendo item 02
Rode uma regularização ridge com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o  𝑅2 na base de testes. Qual o melhor modelo?

In [2]:
ridge = Pipeline([('prep', pre), ('model', Ridge(alpha=1.0))])
ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)
print("R² Ridge:", r2_score(y_test, pred_ridge))

R² Ridge: 0.35103562489183215


REsolvendo item 03 - 
Faça o mesmo que no passo 2, com uma regressão LASSO. Qual método chega a um melhor resultado?

In [3]:
lasso = Pipeline([('prep', pre), ('model', Lasso(alpha=0.001, max_iter=10000))])
lasso.fit(X_train, y_train)
pred_lasso = lasso.predict(X_test)
print("R² Lasso:", r2_score(y_test, pred_lasso))

R² Lasso: 0.351386705404127


Resolvendo item 04 - 
Rode um modelo stepwise. Avalie o  𝑅2
  na vase de testes. Qual o melhor resultado?

In [4]:
r2_ridge = r2_score(y_test, pred_ridge)
r2_lasso = r2_score(y_test, pred_lasso)
print("Melhor entre Ridge e Lasso:", "Ridge" if r2_ridge > r2_lasso else "Lasso")

Melhor entre Ridge e Lasso: Lasso


Resolvendo item 05
Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?

In [5]:
print("R² Ridge:", r2_ridge)
print("R² Lasso:", r2_lasso)
print("No Lasso, alguns coeficientes podem virar zero.")

R² Ridge: 0.35103562489183215
R² Lasso: 0.351386705404127
No Lasso, alguns coeficientes podem virar zero.


REsolvendo item 06
Partindo dos modelos que você ajustou, tente melhorar o  𝑅2
na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.

In [6]:
lr = Pipeline([('prep', pre), ('model', LinearRegression())])
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print("R² Linear:", r2_score(y_test, pred_lr))

R² Linear: 0.351037598570209


Resolvendo item 07
Ajuste uma árvore de regressão e veja se consegue um  𝑅2
  melhor com ela.

In [7]:
tree = Pipeline([('prep', pre), ('model', DecisionTreeRegressor(max_depth=5, random_state=42))])
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_test)
print("R² Árvore:", r2_score(y_test, pred_tree))

R² Árvore: 0.3597272652220729
